# Leather Defect Segmentation — Attention UNet Fine-Tuning

Improvements over the original notebook:
1. **Attention UNet** architecture with Squeeze-and-Excitation (SE) blocks
2. **Aggressive data augmentation** — handles the tiny 92-sample dataset
3. **Focal + Dice combined loss** — addresses extreme class imbalance
4. **Class weighting** computed from actual pixel distribution
5. **Cosine annealing LR** with warm restarts
6. **Resume from checkpoint** support
7. **Comprehensive evaluation** with per-class metrics

### Cell 1 — Imports & Configuration
Set up the environment, import libraries, and define all global constants.

In [ ]:
import os
import sys
import warnings
import glob
from pathlib import Path

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving plots
import matplotlib.pyplot as plt

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras import backend as K
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")

# ============================================================================
# CONFIGURATION
# ============================================================================
BASE_DIR    = Path(r"d:\7th sem\Image processing and CV\Leather_defect_detection\leather")
RESULTS_DIR = Path(r"d:\7th sem\Image processing and CV\Leather_defect_detection\unet_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE      = (256, 256)
IMG_CHANNELS  = 1
CLASS_NAMES   = ['background', 'color', 'cut', 'fold', 'glue', 'poke']
NUM_CLASSES   = len(CLASS_NAMES)   # 6
BASE_FILTERS  = 32
BATCH_SIZE    = 4
EPOCHS        = 30
LEARNING_RATE = 3e-4
WARMUP_EPOCHS = 5
VAL_SPLIT     = 0.2
SEED          = 42
FOCAL_GAMMA   = 2.0
FOCAL_ALPHA   = 0.25
DICE_SMOOTH   = 1.0
DEFECT_TYPES  = ['color', 'cut', 'fold', 'glue', 'poke']

CLASS_COLORS = np.array([
    [0,   0,   0  ],  # background — black
    [255, 0,   0  ],  # color      — red
    [0,   255, 0  ],  # cut        — green
    [0,   0,   255],  # fold       — blue
    [255, 255, 0  ],  # glue       — yellow
    [255, 0,   255],  # poke       — magenta
], dtype=np.uint8)

print(f"Results dir: {RESULTS_DIR}")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

### Cell 2 — Data Discovery
Scan the MVTec leather dataset directory for image–mask pairs per defect class.

In [ ]:
def discover_pairs():
    """Discover all (image_path, mask_path, class_id) triples from the dataset."""
    pairs = []

    for class_idx, defect_type in enumerate(DEFECT_TYPES, start=1):
        test_dir = BASE_DIR / "test" / defect_type
        gt_dir   = BASE_DIR / "ground_truth" / defect_type

        if not test_dir.exists() or not gt_dir.exists():
            print(f"  WARNING: Missing directory for {defect_type}")
            continue

        images = sorted(test_dir.glob("*.png"))
        for img_path in images:
            mask_path = gt_dir / f"{img_path.stem}_mask.png"
            if mask_path.exists():
                pairs.append((str(img_path), str(mask_path), class_idx))

    print(f"Found {len(pairs)} image-mask pairs across {len(DEFECT_TYPES)} defect types")
    class_counts = {}
    for _, _, cls_id in pairs:
        name = CLASS_NAMES[cls_id]
        class_counts[name] = class_counts.get(name, 0) + 1
    for name, count in class_counts.items():
        print(f"  {name}: {count} images")

    return pairs


pairs = discover_pairs()

### Cell 3 — Data Loading & Augmentation
Define preprocessing and augmentation helpers, then wrap them in a `tf.data` generator pipeline.

In [ ]:
def load_and_preprocess(img_path, mask_path, class_id):
    """Load a grayscale image and produce a multiclass mask."""
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=-1)  # (H, W, 1)

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, IMG_SIZE, interpolation=cv2.INTER_NEAREST)

    multiclass_mask = np.zeros(IMG_SIZE, dtype=np.int32)
    multiclass_mask[mask > 127] = class_id
    return img, multiclass_mask


def augment_pair(image, mask):
    """Aggressive augmentation for a tiny dataset."""
    if np.random.rand() > 0.5:
        image, mask = np.fliplr(image), np.fliplr(mask)
    if np.random.rand() > 0.5:
        image, mask = np.flipud(image), np.flipud(mask)

    k = np.random.randint(0, 4)
    image, mask = np.rot90(image, k), np.rot90(mask, k)

    if np.random.rand() > 0.5:
        image = np.clip(image * np.random.uniform(0.7, 1.3), 0.0, 1.0)

    if np.random.rand() > 0.5:
        factor = np.random.uniform(0.7, 1.3)
        mean = np.mean(image)
        image = np.clip((image - mean) * factor + mean, 0.0, 1.0)

    if np.random.rand() > 0.5:
        noise = np.random.normal(0, 0.02, image.shape).astype(np.float32)
        image = np.clip(image + noise, 0.0, 1.0)

    if np.random.rand() > 0.5:
        scale = np.random.uniform(0.85, 1.15)
        h, w = image.shape[:2]
        new_h, new_w = int(h * scale), int(w * scale)
        image_s = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        if len(image_s.shape) == 2:
            image_s = np.expand_dims(image_s, -1)
        mask_s = cv2.resize(mask.astype(np.float32), (new_w, new_h),
                            interpolation=cv2.INTER_NEAREST).astype(np.int32)
        if scale > 1.0:
            sh, sw = (new_h - h) // 2, (new_w - w) // 2
            image = image_s[sh:sh+h, sw:sw+w]
            mask  = mask_s[sh:sh+h, sw:sw+w]
        else:
            ph, pw = (h - new_h) // 2, (w - new_w) // 2
            image_new = np.zeros((h, w, 1), dtype=np.float32)
            mask_new  = np.zeros((h, w), dtype=np.int32)
            image_new[ph:ph+new_h, pw:pw+new_w] = image_s
            mask_new[ph:ph+new_h, pw:pw+new_w]  = mask_s
            image, mask = image_new, mask_new

    if len(image.shape) == 2:
        image = np.expand_dims(image, -1)
    return image.astype(np.float32), mask.astype(np.int32)


def create_dataset(pairs, augment=False, repeat=True):
    """Create a tf.data.Dataset from a list of (img_path, mask_path, class_id) tuples."""
    img_paths  = [p[0] for p in pairs]
    mask_paths = [p[1] for p in pairs]
    class_ids  = [p[2] for p in pairs]

    def generator():
        indices = list(range(len(img_paths)))
        if augment:
            np.random.shuffle(indices)
        for idx in indices:
            img, mask = load_and_preprocess(img_paths[idx], mask_paths[idx], class_ids[idx])
            if augment:
                img, mask = augment_pair(img, mask)
            yield img, mask

    output_sig = (
        tf.TensorSpec(shape=(IMG_SIZE[0], IMG_SIZE[1], IMG_CHANNELS), dtype=tf.float32),
        tf.TensorSpec(shape=(IMG_SIZE[0], IMG_SIZE[1]),               dtype=tf.int32),
    )
    ds = tf.data.Dataset.from_generator(generator, output_signature=output_sig)
    if repeat:
        ds = ds.repeat()
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


print("Data helpers defined.")

### Cell 4 — Class Weights & Train/Val Split
Compute per-class pixel frequency weights to handle the extreme background imbalance, then split the dataset.

In [ ]:
def compute_class_weights(pairs):
    """Inverse-frequency class weights from actual pixel distribution."""
    print("\nComputing class weights from pixel distribution...")
    pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)

    for img_path, mask_path, class_id in pairs:
        _, mask = load_and_preprocess(img_path, mask_path, class_id)
        for c in range(NUM_CLASSES):
            pixel_counts[c] += np.sum(mask == c)

    total = pixel_counts.sum()
    print(f"  Total pixels: {total:,}")
    for i, name in enumerate(CLASS_NAMES):
        print(f"  {name}: {pixel_counts[i]:,} ({pixel_counts[i]/total*100:.2f}%)")

    weights = np.zeros(NUM_CLASSES, dtype=np.float32)
    for i in range(NUM_CLASSES):
        weights[i] = total / (NUM_CLASSES * pixel_counts[i]) if pixel_counts[i] > 0 else 1.0

    weights = np.clip(weights, 0.1, 50.0)
    weights = weights / weights.mean()

    print("\n  Class weights (normalized):")
    for i, name in enumerate(CLASS_NAMES):
        print(f"    {name}: {weights[i]:.4f}")
    return weights


# --- Split & build datasets ---
class_weights = compute_class_weights(pairs)

train_pairs, val_pairs = train_test_split(
    pairs, test_size=VAL_SPLIT, random_state=SEED,
    stratify=[p[2] for p in pairs]
)
print(f"\nTrain: {len(train_pairs)} | Val: {len(val_pairs)}")

train_ds = create_dataset(train_pairs, augment=True,  repeat=True)
val_ds   = create_dataset(val_pairs,   augment=False, repeat=True)

steps_per_epoch = max(len(train_pairs) * 2 // BATCH_SIZE, 1)
val_steps       = max(len(val_pairs)       // BATCH_SIZE, 1)
print(f"Steps per epoch: {steps_per_epoch} | Val steps: {val_steps}")

### Cell 5 — Attention UNet Architecture
Builds the model with:
- **Squeeze-and-Excitation** blocks for channel attention
- **Attention gates** on every skip connection
- **Spatial Dropout** for regularisation

In [ ]:
def squeeze_excite_block(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(max(filters // ratio, 1), activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])


def conv_block(x, filters, dropout_rate=0.1):
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = squeeze_excite_block(x)
    if dropout_rate > 0:
        x = layers.SpatialDropout2D(dropout_rate)(x)
    return x


def attention_gate(x, g, filters):
    """x = skip features, g = gating signal from decoder."""
    theta_x = layers.Conv2D(filters, 1, padding='same')(x)
    phi_g   = layers.Conv2D(filters, 1, padding='same')(g)
    act  = layers.Activation('relu')(layers.Add()([theta_x, phi_g]))
    psi  = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(act)
    return layers.Multiply()([x, psi])


def build_attention_unet(input_shape=(256, 256, 1), num_classes=6, base_filters=32):
    inputs = layers.Input(shape=input_shape, name='input_image')

    # Encoder
    e1 = conv_block(inputs, base_filters,      dropout_rate=0.05)
    e2 = conv_block(layers.MaxPooling2D(2)(e1), base_filters * 2,  dropout_rate=0.10)
    e3 = conv_block(layers.MaxPooling2D(2)(e2), base_filters * 4,  dropout_rate=0.15)
    e4 = conv_block(layers.MaxPooling2D(2)(e3), base_filters * 8,  dropout_rate=0.20)

    # Bottleneck
    b = conv_block(layers.MaxPooling2D(2)(e4), base_filters * 16, dropout_rate=0.25)

    # Decoder
    u4 = layers.Conv2DTranspose(base_filters * 8, 2, strides=2, padding='same')(b)
    d4 = conv_block(layers.Concatenate()([u4, attention_gate(e4, u4, base_filters * 4)]),
                    base_filters * 8, dropout_rate=0.20)

    u3 = layers.Conv2DTranspose(base_filters * 4, 2, strides=2, padding='same')(d4)
    d3 = conv_block(layers.Concatenate()([u3, attention_gate(e3, u3, base_filters * 2)]),
                    base_filters * 4, dropout_rate=0.15)

    u2 = layers.Conv2DTranspose(base_filters * 2, 2, strides=2, padding='same')(d3)
    d2 = conv_block(layers.Concatenate()([u2, attention_gate(e2, u2, base_filters)]),
                    base_filters * 2, dropout_rate=0.10)

    u1 = layers.Conv2DTranspose(base_filters, 2, strides=2, padding='same')(d2)
    d1 = conv_block(layers.Concatenate()([u1, attention_gate(e1, u1, base_filters // 2)]),
                    base_filters, dropout_rate=0.05)

    outputs = layers.Conv2D(num_classes, 1, activation='softmax', name='output')(d1)
    return Model(inputs, outputs, name='attention_unet_leather')


print("Architecture helpers defined.")

### Cell 6 — Loss Functions & Metrics
- **Focal loss**: focuses on hard pixels; suppresses easy background
- **Dice loss**: overlap-based, ignores background class
- **Combined loss**: 0.5 × Focal + 0.5 × Dice

In [ ]:
def focal_loss(y_true, y_pred, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_pred = tf.clip_by_value(tf.reshape(y_pred, [-1, NUM_CLASSES]), 1e-7, 1.0 - 1e-7)
    y_true_oh = tf.one_hot(y_true, NUM_CLASSES)
    ce = -y_true_oh * tf.math.log(y_pred)
    pt = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
    focal_weight = alpha * tf.pow(1.0 - pt, gamma)
    return tf.reduce_mean(tf.reduce_sum(ce, axis=-1) * focal_weight)


def dice_loss_fn(y_true, y_pred, smooth=DICE_SMOOTH):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_pred = tf.reshape(y_pred, [-1, NUM_CLASSES])
    y_true_oh = tf.one_hot(y_true, NUM_CLASSES)
    intersection = tf.reduce_sum(y_true_oh * y_pred, axis=0)
    union = tf.reduce_sum(y_true_oh, axis=0) + tf.reduce_sum(y_pred, axis=0)
    dice_per_class = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - tf.reduce_mean(dice_per_class[1:])  # skip background


def combined_loss(y_true, y_pred):
    return 0.5 * focal_loss(y_true, y_pred) + 0.5 * dice_loss_fn(y_true, y_pred)


def mean_iou_metric(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_pred = tf.cast(tf.argmax(tf.reshape(y_pred, [-1, NUM_CLASSES]), axis=-1), tf.int32)
    cm = tf.cast(tf.math.confusion_matrix(y_true, y_pred, num_classes=NUM_CLASSES), tf.float32)
    diag = tf.linalg.diag_part(cm)
    denom = tf.reduce_sum(cm, 1) + tf.reduce_sum(cm, 0) - diag
    iou   = tf.where(denom > 0, diag / denom, tf.zeros_like(diag))
    valid = tf.cast(denom > 0, tf.float32)
    return tf.math.divide_no_nan(tf.reduce_sum(iou), tf.reduce_sum(valid))


def mean_dice_metric(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_pred = tf.cast(tf.argmax(tf.reshape(y_pred, [-1, NUM_CLASSES]), axis=-1), tf.int32)
    cm = tf.cast(tf.math.confusion_matrix(y_true, y_pred, num_classes=NUM_CLASSES), tf.float32)
    diag  = tf.linalg.diag_part(cm)
    denom = tf.reduce_sum(cm, 1) + tf.reduce_sum(cm, 0)
    dice  = tf.where(denom > 0, 2.0 * diag / denom, tf.zeros_like(diag))
    valid = tf.cast(denom > 0, tf.float32)
    return tf.math.divide_no_nan(tf.reduce_sum(dice), tf.reduce_sum(valid))


print("Loss functions and metrics defined.")

### Cell 7 — Cosine Annealing LR Callback
Linear warmup for the first `WARMUP_EPOCHS`, then cosine decay down to `min_lr`.

In [ ]:
class CosineAnnealingWithWarmup(callbacks.Callback):
    """Cosine annealing LR with linear warmup (TF 2.20 compatible)."""
    def __init__(self, max_lr, warmup_epochs, total_epochs, min_lr=1e-6):
        super().__init__()
        self.max_lr        = max_lr
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.min_lr        = min_lr
        self.lr_history    = []

    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.max_lr * (epoch + 1) / max(self.warmup_epochs, 1)
        else:
            progress = (epoch - self.warmup_epochs) / max(self.total_epochs - self.warmup_epochs, 1)
            lr = self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (1 + np.cos(np.pi * progress))

        try:
            self.model.optimizer.learning_rate.assign(lr)
        except Exception:
            try:
                K.set_value(self.model.optimizer.learning_rate, lr)
            except Exception:
                self.model.optimizer.lr = lr

        self.lr_history.append(lr)
        print(f"  LR: {lr:.6f}", end="")


print("LR scheduler defined.")

### Cell 8 — Visualisation & Evaluation Helpers
Functions for colourising predicted masks, overlaying them on images, plotting training curves, and computing per-class IoU / Dice.

In [ ]:
def colorize_mask(mask):
    h, w = mask.shape
    color = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_id in range(NUM_CLASSES):
        color[mask == cls_id] = CLASS_COLORS[cls_id]
    return color


def visualize_predictions(model, val_pairs, num_samples=6, save_path=None):
    n = min(num_samples, len(val_pairs))
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i in range(n):
        img_path, mask_path, class_id = val_pairs[i]
        img, gt_mask = load_and_preprocess(img_path, mask_path, class_id)
        pred = model.predict(np.expand_dims(img, 0), verbose=0)[0]
        pred_mask = np.argmax(pred, axis=-1)

        axes[i, 0].imshow(img[:, :, 0], cmap='gray')
        axes[i, 0].set_title(f"Input ({CLASS_NAMES[class_id]})", fontsize=10)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(colorize_mask(gt_mask))
        axes[i, 1].set_title("Ground Truth", fontsize=10)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(colorize_mask(pred_mask))
        axes[i, 2].set_title("Prediction", fontsize=10)
        axes[i, 2].axis('off')

        img_rgb = cv2.cvtColor((img[:, :, 0] * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
        overlay = img_rgb.copy()
        pred_color = colorize_mask(pred_mask)
        where = pred_mask > 0
        overlay[where] = (0.5 * overlay[where] + 0.5 * pred_color[where]).astype(np.uint8)
        axes[i, 3].imshow(overlay)
        axes[i, 3].set_title("Overlay", fontsize=10)
        axes[i, 3].axis('off')

    plt.suptitle("Attention UNet — Predictions vs Ground Truth", fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Predictions saved to: {save_path}")
    plt.show()
    plt.close()


def plot_training_history(history, save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history.history['loss'],               label='Train Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'],           label='Val Loss',   linewidth=2)
    axes[0].set_title('Loss', fontsize=13, fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history.history['mean_iou_metric'],     label='Train IoU', linewidth=2)
    axes[1].plot(history.history['val_mean_iou_metric'], label='Val IoU',   linewidth=2)
    axes[1].set_title('Mean IoU', fontsize=13, fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(history.history['mean_dice_metric'],     label='Train Dice', linewidth=2)
    axes[2].plot(history.history['val_mean_dice_metric'], label='Val Dice',   linewidth=2)
    axes[2].set_title('Mean Dice', fontsize=13, fontweight='bold'); axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.suptitle("Training History — Attention UNet", fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Training history saved to: {save_path}")
    plt.show()
    plt.close()


def evaluate_model(model, pairs):
    print("\n" + "=" * 60)
    print("DETAILED EVALUATION")
    print("=" * 60)
    total_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    for img_path, mask_path, class_id in pairs:
        img, gt_mask = load_and_preprocess(img_path, mask_path, class_id)
        pred_mask = np.argmax(model.predict(np.expand_dims(img, 0), verbose=0)[0], axis=-1)
        for tc in range(NUM_CLASSES):
            for pc in range(NUM_CLASSES):
                total_cm[tc, pc] += np.sum((gt_mask == tc) & (pred_mask == pc))

    print(f"\n{'Class':<12} {'IoU':>8} {'Dice':>8} {'Precision':>10} {'Recall':>8}")
    print("-" * 50)
    ious, dices = [], []
    for c in range(NUM_CLASSES):
        tp = total_cm[c, c]
        fp = total_cm[:, c].sum() - tp
        fn = total_cm[c, :].sum() - tp
        iou       = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        dice      = 2 * tp / (2*tp + fp + fn) if (2*tp + fp + fn) > 0 else 0.0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        ious.append(iou); dices.append(dice)
        print(f"{CLASS_NAMES[c]:<12} {iou:>8.4f} {dice:>8.4f} {precision:>10.4f} {recall:>8.4f}")

    print("-" * 50)
    print(f"{'Mean (all)':<12} {np.mean(ious):>8.4f} {np.mean(dices):>8.4f}")
    print(f"{'Mean (defect)':<12} {np.mean(ious[1:]):>8.4f} {np.mean(dices[1:]):>8.4f}")
    return ious, dices


print("Visualisation and evaluation helpers defined.")

### Cell 9 — Build (or Resume) Model
If `best_attention_unet.keras` exists the model resumes from checkpoint; otherwise a fresh Attention UNet is built.

In [ ]:
model_path    = str(RESULTS_DIR / 'best_attention_unet.keras')
initial_epoch = 0

if os.path.exists(model_path):
    print("Resuming from saved checkpoint...")
    model = tf.keras.models.load_model(
        model_path,
        custom_objects={
            'combined_loss':     combined_loss,
            'mean_iou_metric':   mean_iou_metric,
            'mean_dice_metric':  mean_dice_metric,
        }
    )
    initial_epoch = 3  # adjust if needed
    print(f"Resumed model. Starting from epoch {initial_epoch + 1}")
else:
    print("Building Attention UNet from scratch...")
    tf.keras.backend.clear_session()
    model = build_attention_unet(
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], IMG_CHANNELS),
        num_classes=NUM_CLASSES,
        base_filters=BASE_FILTERS,
    )

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=combined_loss,
    metrics=[mean_iou_metric, mean_dice_metric],
)

print(f"Model: {model.name}")
print(f"Total parameters: {model.count_params():,}")
model.summary()

### Cell 10 — Train
Set up callbacks (checkpoint, early stopping, cosine LR) and run `model.fit`.

In [ ]:
cb_list = [
    callbacks.ModelCheckpoint(
        model_path,
        monitor='val_mean_iou_metric', mode='max',
        save_best_only=True, verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor='val_mean_iou_metric', mode='max',
        patience=15, restore_best_weights=True, verbose=1,
    ),
    CosineAnnealingWithWarmup(
        max_lr=LEARNING_RATE,
        warmup_epochs=max(0, WARMUP_EPOCHS - initial_epoch),
        total_epochs=EPOCHS,
    ),
]

print("=" * 60)
print(f"Training epochs {initial_epoch + 1} → {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}  |  Steps/epoch: {steps_per_epoch}")
print(f"Initial LR: {LEARNING_RATE}  |  Loss: Focal + Dice")
print("=" * 60)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    initial_epoch=initial_epoch,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    callbacks=cb_list,
    verbose=1,
)

plot_training_history(
    history,
    save_path=str(RESULTS_DIR / 'training_history.png')
)

### Cell 11 — Evaluate & Visualise Predictions
Load the best saved checkpoint, run per-class evaluation, and save prediction grids.

In [ ]:
print("Loading best model for evaluation...")
if os.path.exists(model_path):
    model = tf.keras.models.load_model(
        model_path,
        custom_objects={
            'combined_loss':    combined_loss,
            'mean_iou_metric':  mean_iou_metric,
            'mean_dice_metric': mean_dice_metric,
        }
    )
    print("Best model loaded.")

ious, dices = evaluate_model(model, val_pairs)

visualize_predictions(
    model, val_pairs,
    num_samples=min(8, len(val_pairs)),
    save_path=str(RESULTS_DIR / 'predictions.png')
)

visualize_predictions(
    model, train_pairs[:8],
    num_samples=8,
    save_path=str(RESULTS_DIR / 'train_predictions.png')
)

print(f"\nResults saved to: {RESULTS_DIR}")
print(f"Best model at:    {model_path}")